# Wczytywanie danych do ramki danych pandas

In [ ]:
import pandas as pd
from pathlib import Path


# Względna ścieżka do pliku JSON z ofertami pracy
# input_path = Path(input("Podaj względna ściężkę do pliku .json z ofertami pracy"))

# Sztywno zapisana ścieżka do pliku .json, tak aby nie wpisywać ciągle u góry scieżki
input_path = Path("../scrapping-worker/justjoinit/offers.json")

# Wczytanie danych do ramki danych pandas (na tym etapie dane są "nieczyste")
df: pd.DataFrame = pd.read_json(
    input_path,
    orient="records"
)

# Podgląd pierwszych wierszy DataFrame
df.head(5)

,Details,Tech stack,Salary
0,"{'Type of work': 'Full-time', 'Experience': 'M...","{'Polish': 'C2', 'English': 'B2', 'ETL': 'regu...",Undisclosed salary
1,"{'Type of work': 'Full-time', 'Experience': 'S...","{'Polish': 'C2', 'PL/SQL': 'master', 'SQL': 'm...",Undisclosed salary
2,"{'Type of work': 'Full-time', 'Experience': 'M...","{'PySpark': 'regular', 'Power BI': 'regular', ...",Undisclosed Salary
3,"{'Type of work': 'Full-time', 'Experience': 'S...","{'English': 'C1', 'AWS': 'advanced', 'Airflow'...",Undisclosed salary
4,"{'Type of work': 'Full-time', 'Experience': 'M...","{'Polish': 'C2', 'English': 'B2', 'MS SQL Serv...",Undisclosed salary


# Rozwijanie zagnieżdzionch słowników

### W niektórych kolumnach, takich jak "Details" czy "Tech Stack", mogą znajdować się zagnieżdżone słowniki 
### Aby uzyskać z nich dodatkowe atrybuty jako osobne kolumny w DataFrame, można je „rozwinąć” za pomocą pd.apply(pd.Series)


In [10]:
# UWAGA:
# Jeśli uruchomisz tę komórkę wielokrotnie bez ponownego wczytania danych,
# zmienna 'df' nie będzie już zawierać kolumn z listy 'columns_to_flatten'.
# W takim przypadku należy ponownie uruchomić komórkę wczytującą dane z pliku .json.

def flatten_columns(df: pd.DataFrame, col_label: str) -> pd.DataFrame:
    """
    Rozwija słowniki zawarte w kolumnie `col_label` ramki danych `df`
    i zwraca nowy DataFrame z rozwiniętymi kolumnami.
    """
    expanded_df = df[col_label].apply(pd.Series)
    
    expanded_df.fillna("Not included", inplace=True)
    
    df.drop(labels=col_label, axis=1, inplace=True)

    return expanded_df

def flatten_multiple_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Rozwija wiele kolumn zawierających słowniki i łączy je z oryginalnym DataFrame.
    """
    flattened_dataframes = []
    for col_label in columns:
        flatten_df = flatten_columns(df, col_label)
        flattened_dataframes.append(flatten_df)

    return pd.concat([df] + flattened_dataframes, axis=1)

# Lista kolumn do rozwinięcia
columns_to_flatten: list[str] = ["Details", "Tech stack"]

# Rozwijanie wskazanych kolumn i łączenie z oryginalnym DataFrame
offers_df: pd.DataFrame = flatten_multiple_columns(df, columns_to_flatten)

# Pobierz listę kolumn z DataFrame
offers_cols: list[str] = offers_df.columns.to_list()

# Popraw nazwę kolumny, jeśli występuje literówka
if ", Prometheus" in offers_cols:
    offers_cols[offers_cols.index(", Prometheus")] = "Prometheus"

# Zaaktualizuj nazwy kolumn
offers_df.columns = offers_cols

offers_df.head()

,Salary,Type of work,Experience,Employment Type,Operating mode,Polish,English,ETL,ETL tools,Informatica,...,Microsoft Copilot,data governance,BCBS239,Data Lineage,Collibra,Jira/Confluence,Microsoft Office,Prometheus,PMM,Linux server systems
0,Undisclosed salary,Full-time,Mid,B2B,Remote,C2,B2,regular,regular,regular,...,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
1,Undisclosed salary,Full-time,Senior,B2B,Hybrid,C2,Not included,Not included,Not included,Not included,...,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
2,Undisclosed Salary,Full-time,Mid,"Permanent, B2B",Hybrid,Not included,Not included,Not included,Not included,Not included,...,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
3,Undisclosed salary,Full-time,Senior,B2B,Remote,Not included,C1,Not included,Not included,Not included,...,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included
4,Undisclosed salary,Full-time,Mid,"B2B, Permanent",Remote,C2,B2,Not included,Not included,Not included,...,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included,Not included


In [ ]:
# Stopień znajomości technologii
advance_levels: dict[str, int] = {         
"Not included":0,
"Nice To Have":1,
"Junior":2,
"Regular":3,
"Advanced":4,
"Master":5
} 

array(['Undisclosed salary', 'Undisclosed Salary'], dtype=object)